## Setup

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from pprintpp import pprint
from functions.get_flight_info import get_flight_info
from mocks.get_flight_info import get_flight_info as mock_get_flight_info
import os
import json

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_KEY"))

In [2]:
tools = [
    {
        "type": "function",
        "function": get_flight_info,
    }
]

### Generic OpenAI query

In [3]:
response = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "user", "content": "When's the next flight from Amsterdam to New York?"}
    ],
)

pprint(dict(response))
# pprint(response.choices[0].message)
# I don’t have real-time flight data ... 

{
    'choices': [
        Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I don’t have access to real-time flight data, so I can’t provide the current next flight schedule from Amsterdam to New York. However, flights between Amsterdam Schiphol Airport (AMS) and New York (JFK or EWR) typically operate multiple times daily, operated by airlines such as KLM, Delta, United, and American Airlines.\n\nTo find the next available flight, please check:\n\n- Airline websites (e.g., KLM, Delta)\n- Flight aggregators (e.g., Google Flights, Expedia, Skyscanner)\n- Amsterdam Schiphol Airport’s official departure schedule\n\nIf you tell me your intended date and preferences (airport in New York, direct/connecting flights), I can guide you on how to look for specific options!', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)),
    ],
    'created': 1769126643,
    'id': 'chatcmpl-D0ywdnRSvfCb7WQpVDesBZTTCEBJC

### Singular function call

In [4]:
response = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "user", "content": "When's the next flight from Amsterdam to New York?"}
    ],
    functions=[get_flight_info],
    function_call="auto",
)

pprint(dict(response))
# {"arguments":"{\"loc_origin\":\"AMS\",\"loc_destination\":\"JFK\"}", "name":"get_flight_info"}

{
    'choices': [
        Choice(finish_reason='function_call', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=FunctionCall(arguments='{"loc_origin":"AMS","loc_destination":"JFK"}', name='get_flight_info'), tool_calls=None)),
    ],
    'created': 1769126651,
    'id': 'chatcmpl-D0ywlI6lUc4Q8U1KgF8qr9CBE5S9V',
    'model': 'gpt-4.1-2025-04-14',
    'object': 'chat.completion',
    'service_tier': 'default',
    'system_fingerprint': 'fp_1a2c4a5ede',
    'usage': CompletionUsage(completion_tokens=23, prompt_tokens=83, total_tokens=106, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)),
}


In [5]:
# parse and pretty print the section of the response that contains the function call
pprint(dict(response.choices[0].message.function_call))

{
    'arguments': '{"loc_origin":"AMS","loc_destination":"JFK"}',
    'name': 'get_flight_info',
}


### Multiple tool calls

Use ```tools``` instead of ```functions```, this is a more general use case and allows multiple tools to be executed in a single query

In [6]:
response = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "user", "content": "When's the next flight from Amsterdam to New York? and also when's the next flight from New York to Amsterdam?"}
    ],
    tools=tools,
    tool_choice="auto",
)

pprint(dict(response.choices[0].message))


{
    'annotations': [],
    'audio': None,
    'content': None,
    'function_call': None,
    'refusal': None,
    'role': 'assistant',
    'tool_calls': [
        ChatCompletionMessageFunctionToolCall(id='call_F4BEsI6XcCF4lSPfx9cABIxE', function=Function(arguments='{"loc_origin": "AMS", "loc_destination": "JFK"}', name='get_flight_info'), type='function'),
        ChatCompletionMessageFunctionToolCall(id='call_DeWobu2I7efcmNLr8QSeRSpe', function=Function(arguments='{"loc_origin": "JFK", "loc_destination": "AMS"}', name='get_flight_info'), type='function'),
    ],
}


Now we get 2 calls to the function

In [7]:
pprint(dict(response.choices[0].message.tool_calls[0]))
pprint(dict(response.choices[0].message.tool_calls[1]))

{
    'function': Function(arguments='{"loc_origin": "AMS", "loc_destination": "JFK"}', name='get_flight_info'),
    'id': 'call_F4BEsI6XcCF4lSPfx9cABIxE',
    'type': 'function',
}
{
    'function': Function(arguments='{"loc_origin": "JFK", "loc_destination": "AMS"}', name='get_flight_info'),
    'id': 'call_DeWobu2I7efcmNLr8QSeRSpe',
    'type': 'function',
}


### Carry on a conversation

In [8]:
messages = [
    {"role": "user", "content": "When's the next flight from Amsterdam to New York? and also the next flight from Chicago to New Jersey?"}
]

assistant_message = client.chat.completions.create(
    model="gpt-4.1",
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

assistant_message = json.loads(assistant_message.model_dump_json())
assistant_message = assistant_message["choices"][0]["message"]


pprint(assistant_message)

{
    'annotations': [],
    'audio': None,
    'content': None,
    'function_call': None,
    'refusal': None,
    'role': 'assistant',
    'tool_calls': [
        {
            'function': {
                'arguments': '{"loc_origin": "AMS", "loc_destination": "JFK"}',
                'name': 'get_flight_info',
            },
            'id': 'call_uIlOsVYYEM6jqribhOZXjdKI',
            'type': 'function',
        },
        {
            'function': {
                'arguments': '{"loc_origin": "ORD", "loc_destination": "EWR"}',
                'name': 'get_flight_info',
            },
            'id': 'call_hDQD13KV85IiE8trlZYhMwRX',
            'type': 'function',
        },
    ],
}


#### Make sure to append the LLM responses to our chat history

In [9]:
messages.append(assistant_message)

In [10]:
if assistant_message["tool_calls"]:
    for tool_call in assistant_message["tool_calls"]:
        if tool_call["type"] == "function":
            function_name = tool_call["function"]["name"]
            function_args = tool_call["function"]["arguments"]
            
            # simulate a python function execution with the arguments specified by our LLM
            tool_eval = eval(f"mock_{function_name}({function_args})")
            
            tool_message = {
                "role": "tool",
                "content": json.dumps(tool_eval),
                "tool_call_id": tool_call["id"],  
            }
            
            messages.append(tool_message)

pprint(messages)

[
    {
        'content': "When's the next flight from Amsterdam to New York? and also the next flight from Chicago to New Jersey?",
        'role': 'user',
    },
    {
        'annotations': [],
        'audio': None,
        'content': None,
        'function_call': None,
        'refusal': None,
        'role': 'assistant',
        'tool_calls': [
            {
                'function': {
                    'arguments': '{"loc_origin": "AMS", "loc_destination": "JFK"}',
                    'name': 'get_flight_info',
                },
                'id': 'call_uIlOsVYYEM6jqribhOZXjdKI',
                'type': 'function',
            },
            {
                'function': {
                    'arguments': '{"loc_origin": "ORD", "loc_destination": "EWR"}',
                    'name': 'get_flight_info',
                },
                'id': 'call_hDQD13KV85IiE8trlZYhMwRX',
                'type': 'function',
            },
        ],
    },
    {
        'content': '{

#### After the calls are served, the LLM will continue the conversation providing a follow-up response to our initial natural language query!

In [11]:
response = client.chat.completions.create(
    model="gpt-4.1",
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

pprint(dict(response))

{
    'choices': [
        Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The next available flights are:\n\n- From Amsterdam (AMS) to New York (JFK): Departs at 10:00 AM with American Airlines, nonstop flight (Flight AA1234), duration 3 hours 30 minutes.\n- From Chicago (ORD) to New Jersey (EWR): Departs at 10:00 AM with American Airlines, nonstop flight (Flight AA1234), duration 3 hours 30 minutes.\n\nIf you need more flight options or details, let me know!', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)),
    ],
    'created': 1769126658,
    'id': 'chatcmpl-D0ywssCMTxRGl1ZU2wakljTl4je32',
    'model': 'gpt-4.1-2025-04-14',
    'object': 'chat.completion',
    'service_tier': 'default',
    'system_fingerprint': 'fp_d38c7f4fa7',
    'usage': CompletionUsage(completion_tokens=102, prompt_tokens=312, total_tokens=414, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tok